# Troubleshoot homomer design and assembly

If structure validation fails, here are suggested steps to take to tune either the CG model or the simulations to generate a successfully validated target assembly.




In [1]:
from itertools import combinations
from pathlib import Path
import json
import numpy as np

from ionerdss.model import pdb
from ionerdss.model.pdb import PDBModelBuilder


## 1. Local NERDSS Executable

This tutorial requires a local NERDSS installation:



In [2]:
# Replace with your actual NERDSS path!
nerdss_dir = Path('~/Workspace/Reaction_ode/nerdss_development').expanduser()


## 2. Specify PDB id and output directory for CG and sim files



In [3]:
pdb_id = "8IP3"
prefix_name = f"{pdb_id}_validation_"

output_dir = Path('~/Workspace/ioNERDSS/TESTS/troubleshoot').expanduser()
output_dir.mkdir(exist_ok=True)

## 3. List all hyperparameters used for Coarse-graining, at their default values, so its clear what options we can use to tune the structure




In [4]:
interface_detect_distance_cutoff=0.6
interface_detect_n_residue_cutoff = 4
min_chain_length = 4

#interface detection
homotypic_detection="auto"
homotypic_detection_interface_radius = 8.0
homotypic_detection_residue_similarity_threshold=0.5
homodimer_distance_threshold=0.5

#chain grouping and alignment
chain_grouping_seq_threshold=0.5
chain_grouping_matching_mode="default"
chain_grouping_custom_aligner="None"

#geometry and regularization
is_on_sphere = "False"
steric_clash_mode="off"
signature_precision=6
template_regularization_strength=0
#Interface homotypic?
interface_type_assignment_distance_threshold=2.0 #\delta_d
interface_type_assignment_angle_threshold=0.2 #\delta_rad

generate_visualizations=True


## 4. Run the Coarse-graining and input file generation steps

In [5]:
# Use output_dir instead of tempfile
sim_workspace = output_dir / f"{pdb_id}_validation"
sim_workspace.mkdir(exist_ok=True)

builder = PDBModelBuilder(source=pdb_id)
system = builder.build_system(
    workspace_path=str(sim_workspace),   # <-- use your dir here
    interface_detect_distance_cutoff=interface_detect_distance_cutoff,
    interface_detect_n_residue_cutoff=interface_detect_n_residue_cutoff,
    interface_type_assignment_distance_threshold=interface_type_assignment_distance_threshold,
    interface_type_assignment_angle_threshold=interface_type_assignment_angle_threshold,
    generate_visualizations=generate_visualizations,
)

2026-04-03 15:03:02 - ionerdss.pdb.8IP3 - WARNING - WARNING: small sigma values : 0.359193; consider increasing binding radius threshold


## 5. Report the number of chains found and the number of interfaces detected


In [ ]:
#Return basic information about the CG model that was built, so the user can check if it matches their expectations.
#print(builder)
print(system)
print(f"Number of chains found N_chain: {len(system.molecule_instances)}")
print(f"Number of distinct subunits found N_mol: {len(system.molecule_types)}")
print(f"Number of total interfaces detected N_interface: {len(system.interface_instances)}")
print(f"Number of unique interfaces detected N_interface_unique: {len(system.interface_types)}")

System(pdb_id=8IP3, molecule_types=1, molecule_instances=5, interface_types=2, interface_instances=10)
Number of chains found N_chain: 5
Number of distinct subunits found N_mol: 1
Number of total interfaces detected N_interface: 10
Number of unique interfaces detected N_interface_unique: 2


## 5a. Inspect designed CG coordinates and interface assignments

In [10]:
# Report the full designed coarse-grained assembly coordinates and interface identities.
pdb.validation.get_designed_structure(system)

{'A_A': {'instance': 'A_A',
  'type': 'A',
  'global_com_coord': (12.601520538330078,
   13.84512939453125,
   10.720112609863282),
  'interfaces': [{'interface_instance': 'A_A_B_A_1',
    'interface_type': 'AA1f',
    'binding_partner_instance': 'B_A',
    'binding_partner_type': 'A',
    'global_interface_coord': (12.837307929992676,
     11.862665176391602,
     15.848976135253906)},
   {'interface_instance': 'A_A_E_A_1',
    'interface_type': 'AA1b',
    'binding_partner_instance': 'E_A',
    'binding_partner_type': 'A',
    'global_interface_coord': (12.406908988952637,
     11.49839973449707,
     15.806063652038574)}]},
 'B_A': {'instance': 'B_A',
  'type': 'A',
  'global_com_coord': (10.578067779541016,
   13.118724060058593,
   10.720111083984374),
  'interfaces': [{'interface_instance': 'B_A_A_A_1',
    'interface_type': 'AA1b',
    'binding_partner_instance': 'A_A',
    'binding_partner_type': 'A',
    'global_interface_coord': (12.749773025512695,
     12.208517074584961,
 

## DEBUGGING STRATEGIES FOR THE CG MODEL:

In [ ]:
# If there are too few interfaces, and some subunits are not connected to the rest of the complex
# then we can try to relax the interface detection parameters and see if we can get more interfaces that connect the complex together.
interface_detect_distance_cutoff=1.0
interface_detect_n_residue_cutoff = 3

# Now rerun the CG model building with the relaxed parameters, and see if we can get more interfaces that connect the complex together.

builder = PDBModelBuilder(source=pdb_id)
system = builder.build_system(
    workspace_path=str(sim_workspace), 
    interface_detect_distance_cutoff=interface_detect_distance_cutoff,
    interface_detect_n_residue_cutoff = interface_detect_n_residue_cutoff,
    interface_type_assignment_distance_threshold=interface_type_assignment_distance_threshold,
    interface_type_assignment_angle_threshold=interface_type_assignment_angle_threshold,
    generate_visualizations=False,
    generate_nerdss_files=False,
)

#And report the statistics of the new model.


In [ ]:
#report statistics of the new model
print(f"Number of chains found N_chain: {len(system.molecule_instances)}")
print(f"Number of distinct subunits found N_mol: {len(system.molecule_types)}")
print(f"Number of total interfaces detected N_interface: {len(system.interface_instances)}")
print(f"Number of unique interfaces detected N_interface_unique: {len(system.interface_types)}")


Number of chains found N_chain: 5
Number of distinct subunits found N_mol: 1
Number of total interfaces detected N_interface: 10
Number of unique interfaces detected N_interface_unique: 2


In [ ]:
# Report coordinates of the rebuilt model after relaxing interface detection.
summarize_designed_structure(system)


In [ ]:
# Print out COM-COM distances for bound partners and the shortest pair in the assembly.
report_bound_distances(system)


## If the homomer keeps assembling well past its target size (a 5-mer is reporting a 12-mer, for example)

This is more likely to indicate that too many interfaces are present.
1. Try making interface detection constrains tighter
2. Try increasing overlapSepLimit, if simulations might be allowing multiple imperfectly aligned subunits to add


In [ ]:
interface_detect_distance_cutoff=0.6 #don't put a stupid comma here or it thinks its a tuple.
interface_detect_n_residue_cutoff = 4

#its possible it is not properly detecting that an interface is the same on both proteins, so relax those constraints?
#Interface homotypic?
interface_type_assignment_distance_threshold=3.0 #\delta_d
interface_type_assignment_angle_threshold=0.3 #\delta_rad

# Now rerun the CG model building with the relaxed parameters, and see if we can get more interfaces that connect the complex together.

builder = PDBModelBuilder(source=pdb_id)
system = builder.build_system(
    workspace_path=str(sim_workspace), 
    interface_detect_distance_cutoff=interface_detect_distance_cutoff,
    interface_detect_n_residue_cutoff = interface_detect_n_residue_cutoff,
    interface_type_assignment_distance_threshold=interface_type_assignment_distance_threshold,
    interface_type_assignment_angle_threshold=interface_type_assignment_angle_threshold,
    generate_visualizations=False,
    generate_nerdss_files=False,
)

#And report the statistics of the new model.

## 6.  list all default NERDSS simulation parameters

In [ ]:
generate_nerdss_files=False,
nerdss_total_molecule_count=100
nerdss_water_box=(100,100,100)
nerdss_time_step=0.1
nerdss_n_itr=100000
default_on_rate_3d_ka=120
default_off_rate_3d_kd=0 #make it irreversible
pdbWrite = 100 #save structures often for easier visualization
timeWrite = 100 #save time series data often for easier tracking of growth.

#RestartWrite must be LESS THAN total iterations, so these files get written!

restartWrite = 20000.0 #save restart files often so we can restart from different points in the trajectory if we want to test different parameters.
if(restartWrite >= nerdss_n_itr):
    raise ValueError("restartWrite must be less than total iterations (nerdss_n_itr) to ensure restart files get written!")

checkPoint= 10000.0
trajWrite = 20000.0
#checkPoint should be more frequent than restart (or equal to it, otherwise restart files won't get writte)
if(checkPoint > restartWrite):
    raise ValueError("checkPoint should be more or equally frequent than restartWrite to ensure restart files get written!")
    #checkPoint = restartWrite. <--- do this


#the titration rate can be sped up to get more copies into the volume over the simulation time.
titration_on_rate=5e-2 #M^-1 s^-1, this is a very slow on rate to make it easier to see the binding events in the trajectory. Adjust as needed.

#overlapSepLimit is a distance threshold in nm that is only evaluated after a subunit associates with another subunit. (they have to be flagged for this whcih is the default)
#it is evaluated between all subunits within the two associating complexes, not just the two subunits that are literally associating, since they may be part of large complexes.
#the maximum value for overlapSepLimit is the shortest COM-COM distance between two bound molecules in the system.
#if it exceeds this value, those otherwise correct binding events will be rejected due to steric clashes. So, it should be set to a value smaller than the shortest COM-COM distance between two bound molecules in the system.
overlapSepLimit = 0.5 # in nm: how close can two bound molecules get within a complex before we consider it a steric clash and reject the move


## 7. Set-up the NERDSS validation simulation, single copies + titration

We use a moderately small box and a longer run.
Use only one copy of each subunit.
Perform titration of additional subunits, to provide more copies to assemble. This titration is essential for homomers.


In [ ]:
import shutil

sim_workspace = output_dir / f"{pdb_id}_validation"

# Clear and recreate fresh each time by uncommenting code below
#if sim_workspace.exists():
#    shutil.rmtree(sim_workspace)
#sim_workspace.mkdir(parents=True)

artifacts = pdb.validation.setup_simulation(
    system,
    workspace_manager=builder.workspace_manager,
    box_nm=nerdss_water_box,
    initial_molecule_count=1,
    titration_on_rate=titration_on_rate,
    parms_overrides={
        'nItr': nerdss_n_itr,
        'timeWrite': timeWrite,
        'timeStep': nerdss_time_step,
        'trajWrite': trajWrite,
        'restartWrite': restartWrite,
        'checkPoint': checkPoint,
        'pdbWrite': pdbWrite, 
        'overlapSepLimit': overlapSepLimit,
    },
)

with open(artifacts.target_file, 'r', encoding='utf-8') as handle:
    target_payload = json.load(handle)

print('Validation workspace:', builder.workspace_manager.workspace_path)
print('Validation counts:', artifacts.molecule_counts)
print('Target file:', artifacts.target_file)
print('parms.inp:', artifacts.nerdss_files['parms'])
print('parms_titrate.inp:', artifacts.nerdss_files['parms_titrate'])
print('Target payload keys:', target_payload.keys())



2026-04-03 10:43:04 - ionerdss.pdb.8IP3 - WARNING - WARNING: small sigma values : 0.359193; consider increasing binding radius threshold


Validation counts: {'A': 5}
Target file: /Users/margaret/Dropbox/s2026/ioNERDSS/TESTS/troubleshoot/8IP3_validation/nerdss_files/structure_validation_target.json
parms.inp: /Users/margaret/Dropbox/s2026/ioNERDSS/TESTS/troubleshoot/8IP3_validation/nerdss_files/parms.inp
parms_titrate.inp: /Users/margaret/Dropbox/s2026/ioNERDSS/TESTS/troubleshoot/8IP3_validation/nerdss_files/parms_titrate.inp
Target payload keys: dict_keys(['molecule_counts', 'designed_coordinates'])


## 8. Run the Actual NERDSS Validation Simulation

This is the key validation step: the observed structure is taken from a real NERDSS run, not from a synthetic rigid transform.

If no full assembly matching the designed one-copy target appears in the histogram, the validation suite returns a warning and does not attempt alignment.


In [ ]:


simulation_error = None
try:
    sim_result = pdb.validation.run_simulation(
        artifacts,
        nerdss_dir=nerdss_dir,
    )
except ValueError as exc:
    sim_result = None
    simulation_error = exc
    print(exc)


Working directory set to: /Users/margaret/Dropbox/s2026/ioNERDSS/TESTS/troubleshoot/8IP3_validation/nerdss_files


ValueError: The target composition appeared in the histogram, but no matching connected assembly was found in DATA/restart.dat or any RESTART snapshot. No matching connected assembly was found in DATA/restart.dat or any RESTART snapshot. restart.dat: No connected component in the final restart snapshot matches the designed target composition {'A': 5}.

In [ ]:
# now check the validation results and report on the growth of the complex, and whether it matches the expected behavior based on the CG model that was built.

In [ ]:


if sim_result is None:
    print('Simulation failed before a validation result was returned:')
    print(simulation_error)
else:
    print('Simulation dir:', sim_result.simulation_dir)
    print('Histogram file:', sim_result.histogram_file)
    print('Full assembly found:', sim_result.full_assembly_found)
    print('First full assembly time:', sim_result.first_full_assembly_time)
    if sim_result.warning_message:
        print(sim_result.warning_message)
    else:
        print('Observed coordinates:', sim_result.observed_coordinates)


NameError: name 'sim_result_8erq' is not defined

In [ ]:
# Plot the observed coordinates when a full assembly was successfully extracted.
import matplotlib.pyplot as plt
if sim_result is None or sim_result.observed_coordinates is None:
    print('No observed coordinates to plot.')
else:
    print(sim_result.observed_coordinates)
    coords = np.array(list(sim_result.observed_coordinates.values()), dtype=float)
    fig = plt.figure()
    ax = fig.add_subplot(projection='3d')
    ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2])
    ax.set_title(f'Observed Coordinates for {pdb_id}')
    ax.set_xlabel('X (nm)')
    ax.set_ylabel('Y (nm)')
    ax.set_zlabel('Z (nm)')
    plt.show()



None


## DEBUGGING STRATEGIES--IF A FULL ASSEMBLY IS NOT FOUND:

If a full assembly is not found, try running the simulation for longer. 
For heteromers, you can also slow down titration if it is adding too many copies too quickly, leading to kinetic traps.
If you speed up titration, it will add more monomers of all types more quickly to the volume.
You can also increase the system volume if it is getting too crowded for a very large assembly. 

In [ ]:
nerdss_water_box=(100,100,100)
nerdss_time_step=0.2
nerdss_n_itr=500000
default_on_rate_3d_ka=120
default_off_rate_3d_kd=0 #make it irreversible
pdbWrite = 100 #save structures often for easier visualization
timeWrite = 100 #save time series data often for easier tracking of growth.

#the titration rate can be sped up to get more copies into the volume over the simulation time.
titration_on_rate=5e-4 #M^-1 s^-1, this is a very slow on rate to make it easier to see the binding events in the trajectory. Adjust as needed.


## re-set-up a NERDSS simualtion with new parmaeters.

In [ ]:
artifacts = pdb.validation.setup_simulation(
    system,
    workspace_manager=builder.workspace_manager,
    box_nm=nerdss_water_box,
    initial_molecule_count=1,
    titration_on_rate=titration_on_rate,
    parms_overrides={
        'nItr': nerdss_n_itr,
        'timeWrite': timeWrite,
        'timeStep': nerdss_time_step,
        'trajWrite': trajWrite,
        'restartWrite': restartWrite,
        'checkPoint': checkPoint,
        'pdbWrite': pdbWrite,
        'overlapSepLimit': overlapSepLimit,
    },
)

2026-04-03 10:47:34 - ionerdss.pdb.8IP3 - WARNING - WARNING: small sigma values : 0.359193; consider increasing binding radius threshold


## 6. VALIDATE: Align the Observed NERDSS Structure Back to the Design

Only perform alignment when the real NERDSS run actually produced at least one full assembly.


In [ ]:
if sim_result is not None and sim_result.full_assembly_found and sim_result.observed_coordinates is not None:
    alignment = pdb.validation.align_structure(
        artifacts.designed_coordinates,
        sim_result.observed_coordinates,
        backend='kabsch',
    )
    print('Observed RMSD:', alignment.rmsd)
else:
    print('Skipping alignment because no full assembly coordinates were returned from the NERDSS validation run.')


ValueError: Coordinate arrays must have shape (N, 3).

## More troubleshooting
If it can find a completed assembly, the RMSD should be nice and low.
If not, there may be too many interfaces?
For homomers, there can be other issues

In [ ]:
# we can suggest changing the overlapSepLimit.
